In [5]:
%load_ext autoreload
%autoreload 2
from notebook import *
# if get something about NUMEXPR_MAX_THREADS being set incorrectly, don't worry.  It's not a problem.

### Even with locks...

### Share memory example

In [31]:
compare([do_render_code("vadd_mt.c",show=["// interleaving","#endif"]),do_render_code("vadd_mt.c",show=["CHUNK","#else"])])

In [2]:
! make clean; make vadd_mt_chunk; make vadd_mt

rm -f testloop testloop_O3 coherence blockmm blockmm_pthread value_of_i vadd_sse fence mfence vadd_sse_two_threads vadd_sse_two_threads_chunk vadd_mt vadd_mt_chunk testloop_volatile testloop_O3_volatile coherence_lock
rm -rf *.dSYM
cc -O3 -Wno-format-zero-length -Wno-implicit-function-declaration -mno-avx -DHAVE_LINUX_PERF_EVENT_H -pthread -DSSE -DCHUNK vadd_mt.c perfstats.c -o vadd_mt_chunk 
cc -O3 -Wno-format-zero-length -Wno-implicit-function-declaration -mno-avx -DHAVE_LINUX_PERF_EVENT_H -pthread -DSSE vadd_mt.c perfstats.c -o vadd_mt


In [7]:
! echo "size,threads,IC,Cycles,CPI,CT,ET,L1_dcache_miss_rate,L1_dcache_misses,L1_dcache_accesses,branches,branch_misses" > stats.csv
! echo -n "16777216,4," >> stats.csv
! echo "interleaving:"; cd ~/courses/CSE142/demo/multiprocessor; perf stat -e cycles,instructions,cache-misses ./vadd_mt
! echo -n "16777216,4," >> stats.csv
! echo "chunking:"; cd ~/courses/CSE142/demo/multiprocessor; perf stat -e cycles,instructions,cache-misses  ./vadd_mt_chunk

interleaving:
67108864	67108864	67108864	67108864	Takes 0.318677 seconds
132.000000	

 Performance counter stats for './vadd_mt':

    30,160,144,556      cpu_atom/cycles/                                                        (0.92%)
    37,745,235,788      cpu_core/cycles/                                                        (97.66%)
    13,334,476,363      cpu_atom/instructions/           #    0.44  insn per cycle              (0.92%)
    48,910,743,763      cpu_core/instructions/           #    1.30  insn per cycle              (97.66%)
       239,645,899      cpu_atom/cache-misses/                                                  (0.92%)
       181,667,187      cpu_core/cache-misses/                                                  (97.67%)

       6.610042812 seconds time elapsed

       6.674642000 seconds user
       0.709643000 seconds sys


chunking:
67108864	67108864	67108864	67108864	Takes 0.073820 seconds
132.000000	

 Performance counter stats for './vadd_mt_chunk':

  

In [8]:
display_df_mono(render_csv("stats.csv"))

,index,size,threads,IC,Cycles,CPI,CT,ET,L1_dcache_miss_rate,L1_dcache_misses,L1_dcache_accesses,branches,branch_misses
0,0,16777216,4,1637034225,4399262194,2.687337,0.072244,0.317822,0.104662,73340139,700734940,234003639,14161
1,1,16777216,4,401431877,1023762096,2.550276,0.070972,0.072658,0.089639,15407521,171883834,57398360,9611


## Consistency

In [42]:
render_code("fence.c", show=["#ifndef MFENCE","endif"])

// fence.c:13-28 (16 lines)
#ifndef MFENCE
void* modifya(void *z)
{
  while(!go);
  a=1;
  x=b;
  return NULL;
}
void* modifyb(void *z)
{
  while(!go);
  b=1;
  y=a;
  return NULL;
}
#endif

In [43]:
! make clean; make fence

rm -f testloop testloop_O3 coherence blockmm blockmm_pthread value_of_i vadd_sse fence mfence vadd_sse_two_threads vadd_sse_two_threads_chunk vadd_mt vadd_mt_chunk testloop_volatile testloop_O3_volatile coherence_lock
rm -rf *.dSYM
cc -O3 -Wno-format-zero-length -Wno-implicit-function-declaration fence.c -o fence -lpthread


In [44]:
!rm fence.txt;  touch fence.txt
!lscpu | grep "Model name:"
!cd ~/courses/CSE142/demo/multiprocessor; for i in {1..5000}; do ./fence 2>> fence.txt; done; for i in {1..5000}; do ./fence 2>> fence.txt; done;

rm: cannot remove 'fence.txt': No such file or directory
Model name:                           13th Gen Intel(R) Core(TM) i7-13700


In [45]:
! grep "(0, 0)" fence.txt |wc
! grep "(0, 1)" fence.txt |wc
! grep "(1, 1)" fence.txt |wc
! grep "(1, 0)" fence.txt |wc

      5      20      95
   9654   38616  183426
      5      20      95
    336    1344    6384


In [47]:
! rm fence_amd.txt; touch fence_amd.txt
! ssh htseng@blissey "lscpu | grep 'Model name:'; cd ~/courses/CSE142/demo/multiprocessor; for i in {1..10000}; do ./fence 2>> fence_amd.txt; done;"
! grep "(0, 0)" fence_amd.txt |wc
! grep "(0, 1)" fence_amd.txt |wc
! grep "(1, 1)" fence_amd.txt |wc
! grep "(1, 0)" fence_amd.txt |wc

Model name:                           AMD Ryzen 7 5700X 8-Core Processor
      7      28     133
   9988   39952  189772
      0       0       0
      5      20      95


### "mfence" instructions:

An instruction that forces all updates must finish before making progress after this instruction.

In [20]:
render_code("fence.c", show=["#ifdef MFENCE", "#endif"])

// fence.c:41-57 (17 lines)
#ifdef MFENCE
#define _update_var(_v, _x) { _v = (_x); asm("mfence"); }
void* modifya(void *z)
{
//  a=1;
  _update_var(a,1);
  x=b;
  return NULL;
}
void* modifyb(void *z)
{
//  b=1;
  _update_var(b,1);
  y=a;
  return NULL;
}
#endif

In [21]:
!make mfence
!rm mfence.txt; touch mfence.txt
!for i in {1..10000}; do ./fence 2>> mfence.txt; done;
#!ssh htseng@gengar "cd ~/courses/CS203/demo/multiprocessor; for i in {1..3000}; do ./mfence 2>> mfence.txt; done;"

cc -DMFENCE -O3 -Wno-format-zero-length -Wno-implicit-function-declaration fence.c -o mfence -lpthread


In [22]:
! grep "(0, 0)" mfence.txt |wc
! grep "(0, 1)" mfence.txt |wc
! grep "(1, 1)" mfence.txt |wc
! grep "(1, 0)" mfence.txt |wc

      1       4      19
   9924   39696  188556
      0       0       0
     75     300    1425


In [24]:
! rm mfence_amd.txt; touch mfence_amd.txt
! ssh htseng@blissey "cd ~/courses/CSE142/demo/multiprocessor; for i in {1..10000}; do ./mfence 2>> mfence_amd.txt; done;"
! grep "(0, 0)" mfence_amd.txt |wc
! grep "(0, 1)" mfence_amd.txt |wc
! grep "(1, 1)" mfence_amd.txt |wc
! grep "(1, 0)" mfence_amd.txt |wc

      0       0       0
   9996   39984  189924
      0       0       0
      4      16      76
